# KneeXAI — GPU research runner for Kaggle

This notebook runs the repository's **existing** `scripts/run_research.py` on Kaggle GPU. It does not reimplement the metrics, so the resulting CSV follows the same validated contract used by Research Mode.

## Before running

1. In Kaggle Notebook settings, turn **Accelerator → GPU** on.
2. Attach a Kaggle Dataset containing this project source (without `.venv`, `.git`, and model weights is fine).
3. Attach a private Kaggle Dataset containing matching `.pth` and `.onnx` files for every selected model ID. Expected filenames come from `app.config`.
4. For a full run, provide `inputs/test/final_manifest_250.json` with 50 verified test images per KL grade and the original 150 IDs unchanged.
5. Enable Internet only if Kaggle needs to install `quantus` and ONNX packages.

The shared `final_research` profile is defined in Python. Use `LIMIT = 1` only for a smoke check, with a run ID different from the full experiment.

In [ ]:
# Kaggle already provides CUDA-enabled PyTorch when GPU is enabled.
# Do NOT install torch from requirements.txt here: that file is intentionally CPU-oriented.
import sys
!{sys.executable} -m pip install -q quantus==0.6.0 onnx==1.17.0 onnxruntime==1.20.1 opencv-python-headless==4.10.0.84

import torch
assert torch.cuda.is_available(), (
    'CUDA is unavailable. In Kaggle open Notebook options and select Accelerator → GPU, ' 
    'then restart the session and run this cell again.'
)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

# CHANGE THESE THREE PATHS to the Kaggle Dataset slugs that you attached.
PROJECT_SOURCE = Path('/kaggle/input/kneexai-project')
MODELS_SOURCE = Path('/kaggle/input/kneexai-models')

# One model ID, or 'all' to run the four ordinal configurations sequentially.
MODEL_SELECTION = 'resnet50'

# The final 250-image manifest must contain 50 labelled test images per KL grade.
# Relative image paths are resolved from the copied project root.
MANIFEST_SOURCE = PROJECT_SOURCE / 'inputs' / 'test' / 'final_manifest_250.json'

# The runner owns the one final profile: 50/1024 Faithfulness, 8 normalised MaxSensitivity samples.
EXPERIMENT_PROFILE = 'final_research'

# First run: False. To resume, keep this prefix and every setting below unchanged.
RESUME = False
RUN_ID_PREFIX = 'final-xai-250-v1'  # Change only before a new experiment.

LIMIT = None  # Use 1 or 2 only for a technical smoke test, then choose a new run ID.
SEED = 42

sys.path.insert(0, str(PROJECT_SOURCE))
from app.config import MODEL_CONFIGS, RESEARCH_MODEL_IDS
assert MODEL_SELECTION == 'all' or MODEL_SELECTION in RESEARCH_MODEL_IDS
SELECTED_MODELS = RESEARCH_MODEL_IDS if MODEL_SELECTION == 'all' else (MODEL_SELECTION,)
MODEL_FILES = {model_id: [MODEL_CONFIGS[model_id]['checkpoint_path'].name,
                          MODEL_CONFIGS[model_id]['onnx_path'].name] for model_id in SELECTED_MODELS}
assert PROJECT_SOURCE.is_dir(), f'Project dataset not found: {PROJECT_SOURCE}'
assert MODELS_SOURCE.is_dir(), f'Model dataset not found: {MODELS_SOURCE}'
assert MANIFEST_SOURCE.is_file(), f'Manifest not found: {MANIFEST_SOURCE}'
assert EXPERIMENT_PROFILE == 'final_research'
print('Selected models:', SELECTED_MODELS)
print('Experiment profile:', EXPERIMENT_PROFILE)
print('Manifest:', MANIFEST_SOURCE)

### Manifest note

The runner validates every image before calculation. If your real manifest is in a separate Kaggle dataset, either use **absolute Kaggle paths** in `image_path`, or copy its images under the project directory and use paths relative to it. Do not create labels from directory names.

In [ ]:
import shutil

PROJECT_DIR = Path('/kaggle/working/knee-osteoarthritis-xai')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(PROJECT_SOURCE, PROJECT_DIR, ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__'))

models_dir = PROJECT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
for model_id in SELECTED_MODELS:
    for filename in MODEL_FILES[model_id]:
        source = MODELS_SOURCE / filename
        if not source.is_file():
            raise FileNotFoundError(f'Missing required {model_id} model file: {source}')
        shutil.copy2(source, models_dir / filename)

# A source manifest inside the project keeps its relative paths valid after copying.
# External absolute paths remain valid because Kaggle input datasets are mounted read-only.
if MANIFEST_SOURCE.is_relative_to(PROJECT_SOURCE):
    manifest_path = PROJECT_DIR / MANIFEST_SOURCE.relative_to(PROJECT_SOURCE)
else:
    manifest_path = MANIFEST_SOURCE

print('Project staged at:', PROJECT_DIR)
print('Results will be written under: /kaggle/working')

In [ ]:
# The repository runner owns the shared final profile and writes one CSV per model.
import subprocess

command = [
    sys.executable, 'scripts/run_research.py',
    '--manifest', str(manifest_path),
    '--profile', EXPERIMENT_PROFILE,
    '--output-dir', '/kaggle/working',
    '--seed', str(SEED),
    '--device', 'cuda',
]
if MODEL_SELECTION == 'all':
    command.append('--all-models')
    command += ['--run-id', RUN_ID_PREFIX]
    output_paths = [Path('/kaggle/working') / f'{RUN_ID_PREFIX}-{model_id}.csv' for model_id in SELECTED_MODELS]
else:
    selected_run_id = f'{RUN_ID_PREFIX}-{MODEL_SELECTION}'
    command += ['--model', MODEL_SELECTION, '--run-id', selected_run_id]
    output_paths = [Path('/kaggle/working') / f'{selected_run_id}.csv']
if LIMIT is not None:
    command += ['--limit', str(LIMIT)]
if RESUME:
    command.append('--resume')
print('Running:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)

print('Completed:', *output_paths, sep='\n- ' )

In [ ]:
# Inspect and download the validated results.
import pandas as pd

for output_path in output_paths:
    results = pd.read_csv(output_path)
    print(f'\n{output_path.name}: {len(results)} rows')
    display(results.groupby(['metric_category', 'metric_name']).size().rename('rows').reset_index())
    display(results.groupby(['xai_method'])['metric_value'].agg(['count', 'median', 'mean']).reset_index())
    if 'attribution_l2_norm' in results:
        display(results.groupby('xai_method')['attribution_l2_norm'].agg(['median', 'min', 'max']).reset_index())

print('In the right Kaggle panel, use Output → Save Version or download this CSV and place it in results/research/ locally.')